In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load raw data
df = pd.read_csv('marketing_dataset_sample.csv')

print("Raw data loaded")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(df.head(3))

 Data loaded
Shape: (60, 15)
Columns: ['Week', 'FB_Spend', 'GG_Spend', 'Seasonality', 'Promo', 'FB_Adstock', 'GG_Adstock', 'FB_Effect', 'GG_Effect', 'Traffic', 'Sales', 'Clicks', 'Conversions', 'CPC', 'CR']

First 3 rows:
        Week     FB_Spend     GG_Spend  Seasonality  Promo    FB_Adstock  \
0   1/7/2024  4247.240713  5720.741028     1.000000      1   4247.240713   
1  1/14/2024  7704.285838  4899.443222     1.031813      1   9827.906195   
2  1/21/2024  6391.963651  8801.162564     1.062815      0  11305.916750   

     GG_Adstock  FB_Effect  GG_Effect      Traffic       Sales       Clicks  \
0   5720.741028   0.419132   0.566926  2882.392277  356.442465  2882.392277   
1   6615.665531   0.794387   0.636454  3933.723103  432.305673  3933.723103   
2  10785.862220   0.836413   0.823115  4804.874393  426.433121  4804.874393   

   Conversions       CPC        CR  
0   356.442465  3.683781  0.123662  
1   432.305673  4.896307  0.109897  
2   426.433121  3.325770  0.088750  


In [ ]:
print("="*50)
print("DATA QUALITY CHECK")
print("="*50)

# Check missing values
missing = df.isnull().sum()
if missing.sum() > 0:
    print(f"Missing values:\n{missing[missing > 0]}")
else:
    print("No missing values")

# Date range
if 'Week' in df.columns:
    print(f"\nDate range: {df['Week'].iloc[0]} to {df['Week'].iloc[-1]}")
    print(f"Total weeks: {len(df)}")

# Basic statistics
print(f"\nSpend statistics:")
print(f"  FB_Spend: min={df['FB_Spend'].min():.0f}, max={df['FB_Spend'].max():.0f}, mean={df['FB_Spend'].mean():.0f}")
print(f"  GG_Spend: min={df['GG_Spend'].min():.0f}, max={df['GG_Spend'].max():.0f}, mean={df['GG_Spend'].mean():.0f}")

print(f"\nSales: min={df['Sales'].min():.0f}, max={df['Sales'].max():.0f}, mean={df['Sales'].mean():.0f}")

DATA QUALITY CHECK
 No missing values

 Date range: 1/7/2024 to 2/23/2025
   Total weeks: 60

 Spend summary:
   FB_Spend: min=2124, max=7819, mean=4805
   GG_Spend: min=3039, max=9908, mean=6407

 Sales: min=309, max=500, mean=411


In [ ]:
print("="*50)
print("BASIC METRICS")
print("="*50)

# Calculate basic metrics
df['CPC'] = (df['FB_Spend'] + df['GG_Spend']) / (df['Clicks'] + 1e-6)
df['ROAS'] = df['Sales'] / (df['FB_Spend'] + df['GG_Spend'] + 1e-6)
df['ROI'] = (df['Sales'] - df['FB_Spend'] - df['GG_Spend']) / (df['FB_Spend'] + df['GG_Spend'] + 1e-6)
df['CR'] = df['Conversions'] / (df['Clicks'] + 1e-6)

print(f"Avg CPC: {df['CPC'].mean():.0f}")
print(f"Avg ROAS: {df['ROAS'].mean():.2f}")
print(f"Avg ROI: {df['ROI'].mean():.2%}")
print(f"Avg CR: {df['CR'].mean():.2%}")

 Calculated additional metrics
   Avg CPC: 3
   Avg ROAS: 0.04
   Avg ROI: -96.14%


In [ ]:
print("="*50)
print("FEATURE ENGINEERING - PART 1")
print("="*50)

original_cols = len(df.columns)

# Ratio features
df['FB_GG_Ratio'] = df['FB_Spend'] / (df['GG_Spend'] + 1e-6)
df['GG_FB_Ratio'] = df['GG_Spend'] / (df['FB_Spend'] + 1e-6)

# Spend aggregation
df['Total_Spend'] = df['FB_Spend'] + df['GG_Spend']
df['FB_Share'] = df['FB_Spend'] / (df['Total_Spend'] + 1e-6)
df['GG_Share'] = df['GG_Spend'] / (df['Total_Spend'] + 1e-6)

# Efficiency metrics
df['Sales_per_Spend'] = df['Sales'] / (df['Total_Spend'] + 1e-6)
df['Clicks_per_Spend'] = df['Clicks'] / (df['Total_Spend'] + 1e-6)

print(f"Added {len(df.columns) - original_cols} new features")
print(f"New columns: {df.columns.tolist()[-6:]}")

 Saved to 'data_processed_module1.csv'

MODULE 1 - COMPLETED
Total records: 60
Total FB Spend: 288,300
Total GG Spend: 384,422
Total Sales: 24,648


In [ ]:
print("="*50)
print("FEATURE ENGINEERING - PART 2 (LAG)")
print("="*50)

# Lag features (1 week delay)
df['FB_Spend_lag1'] = df['FB_Spend'].shift(1)
df['GG_Spend_lag1'] = df['GG_Spend'].shift(1)
df['Total_Spend_lag1'] = df['Total_Spend'].shift(1)
df['Sales_lag1'] = df['Sales'].shift(1)
df['CPC_lag1'] = df['CPC'].shift(1)

# Lag 2 weeks
df['FB_Spend_lag2'] = df['FB_Spend'].shift(2)
df['GG_Spend_lag2'] = df['GG_Spend'].shift(2)
df['Sales_lag2'] = df['Sales'].shift(2)

print(f"Added lag features (1 week, 2 weeks)")

In [ ]:
print("="*50)
print("FEATURE ENGINEERING - PART 3 (ROLLING)")
print("="*50)

# Rolling averages (4 weeks)
df['FB_Spend_roll4'] = df['FB_Spend'].rolling(4).mean()
df['GG_Spend_roll4'] = df['GG_Spend'].rolling(4).mean()
df['Total_Spend_roll4'] = df['Total_Spend'].rolling(4).mean()
df['Sales_roll4'] = df['Sales'].rolling(4).mean()
df['CPC_roll4'] = df['CPC'].rolling(4).mean()

# Rolling sums (4 weeks)
df['FB_Spend_sum4'] = df['FB_Spend'].rolling(4).sum()
df['GG_Spend_sum4'] = df['GG_Spend'].rolling(4).sum()
df['Total_Spend_sum4'] = df['Total_Spend'].rolling(4).sum()

# Rolling std (volatility)
df['FB_Spend_std4'] = df['FB_Spend'].rolling(4).std()
df['GG_Spend_std4'] = df['GG_Spend'].rolling(4).std()

print(f"Added rolling window features (4 weeks)")

In [ ]:
print("="*50)
print("FEATURE ENGINEERING - PART 4 (INTERACTION)")
print("="*50)

# Interaction with Promo
df['FB_Promo'] = df['FB_Spend'] * df['Promo']
df['GG_Promo'] = df['GG_Spend'] * df['Promo']
df['Total_Spend_Promo'] = df['Total_Spend'] * df['Promo']

# Interaction with Seasonality
df['FB_Season'] = df['FB_Spend'] * df['Seasonality']
df['GG_Season'] = df['GG_Spend'] * df['Seasonality']
df['Total_Spend_Season'] = df['Total_Spend'] * df['Seasonality']

# Cross-channel interaction
df['FB_GG_Interaction'] = df['FB_Spend'] * df['GG_Spend'] / 1000

print(f"Added interaction features")

In [ ]:
print("="*50)
print("HANDLE MISSING VALUES")
print("="*50)

# Check missing before
missing_before = df.isnull().sum().sum()
print(f"Missing values before: {missing_before}")

# Drop rows with NaN (first 4 weeks will have NaN from rolling)
df = df.dropna().reset_index(drop=True)

# Check missing after
missing_after = df.isnull().sum().sum()
print(f"Missing values after: {missing_after}")
print(f"Rows dropped: {missing_before - missing_after}")
print(f"Final shape: {df.shape}")

In [ ]:
print("="*50)
print("FINAL DATASET SUMMARY")
print("="*50)

print(f"Final shape: {df.shape}")
print(f"Total features: {len(df.columns)}")
print(f"Date range: {df['Week'].iloc[0]} to {df['Week'].iloc[-1]}")
print(f"Total weeks: {len(df)}")

print(f"\nFeature list:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col}")

# Save processed data
df.to_csv('data_processed_module1.csv', index=False)
print("\n✅ Saved to 'data_processed_module1.csv'")

print("\n" + "="*50)
print("MODULE 1 - COMPLETED")
print("="*50)
print(f"Total records: {len(df)}")
print(f"Total FB Spend: {df['FB_Spend'].sum():,.0f}")
print(f"Total GG Spend: {df['GG_Spend'].sum():,.0f}")
print(f"Total Sales: {df['Sales'].sum():,.0f}")
print(f"Total features: {len(df.columns)}")

In [ ]:
# Check correlation with target
print("="*50)
print("CORRELATION WITH SALES (Top 10)")
print("="*50)

corr_with_sales = df.corr()['Sales'].sort_values(ascending=False)
print(corr_with_sales.head(10))